# RIG + Globus Transfer: end-to-end tutorial (AmSC PAT → RIG → Globus)

This notebook moves data with Globus **through RIG**, and your code never handles a Globus token. You send your
**AmSC PAT**. RIG takes the project from the PAT (`amsc_project_context`), swaps the PAT for your **project's Globus
Service Account** token, and forwards the request unchanged to the Globus Transfer API (`transfer.api.globus.org`).

```
You --(AmSC PAT)--> Kong --> RIG --(project Globus Service Account token)--> Globus Transfer API
```

The data itself moves between Globus collections (DTN to DTN), never through RIG.

**Steps**

1. Load your AmSC PAT (from [`../login-amsc.ipynb`](../login-amsc.ipynb))
2. Discover your project's Globus collections
3. List files on a collection
4. Copy a file into a new directory on a collection
5. Delete the copy

Every call goes to `<RIG>/globus/external/transfer/v0.10/<Globus Transfer API path>`, plus `<RIG>/globus/internal/collections`
for step 2.

> These are **real** Globus operations on your project's collections: step 4 creates a directory and step 5 deletes it.

### Prerequisites

- An AmSC PAT from **[`../login-amsc.ipynb`](../login-amsc.ipynb)** (saved to `~/.amsc_token.json`). The PAT's project is the project whose
  Globus Service Account and collections are used. To use another project, generate a new PAT.
- A **Globus Service Account provisioned for that project** by the VO / AmSC administrators. You cannot create it yourself.
- At least one **Globus collection registered for the project** in RIG-UI → **Globus Collections** (see step 2), with a
  small non-empty file under its path for the copy step. Registering a second collection lets you copy between them.
- `pip install requests python-dotenv`

## 0) Configuration

Values come from `~/.amsc_token.json` (written by [`../login-amsc.ipynb`](../login-amsc.ipynb)). Environment variables (or `.env`) override them:

| Variable | Meaning | Default |
|---|---|---|
| `AMSC_PAT` | AmSC PAT | `AMSC_PAT` from `~/.amsc_token.json` |
| `AMSC_RIG_URL` | RIG base URL | from `~/.amsc_token.json`, else staging RIG |
| `AMSC_CREDENTIAL_VAULT_URL` | RIG-UI base URL (Globus Collections page) | from `~/.amsc_token.json`, else staging RIG-UI |
| `GLOBUS_SOURCE`, `GLOBUS_DEST` | Collection to copy from / to (`#`, collection UUID, or label) | prompt |
| `GLOBUS_FILENAME` | File to copy (name under the source collection path) | first non-empty file |

For a RIG with a private CA (e.g. the local huck sandbox), point `REQUESTS_CA_BUNDLE` at the CA bundle.

In [1]:
import sys
!{sys.executable} -m pip install requests python-dotenv

In [2]:
import os
import json
import time
import base64
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List

import requests
from dotenv import load_dotenv

load_dotenv()

home_path = Path("~/.amsc_token.json").expanduser()
token_file = json.loads(home_path.read_text()) if home_path.exists() else {}

PAT = os.environ.get("AMSC_PAT") or token_file.get("AMSC_PAT")
RIG_URL = (os.environ.get("AMSC_RIG_URL") or token_file.get("AMSC_RIG_URL") or "https://rig.staging.american-science-cloud.org").rstrip("/")
RIG_UI_URL = (os.environ.get("AMSC_CREDENTIAL_VAULT_URL") or token_file.get("AMSC_CREDENTIAL_VAULT_URL") or "https://rig-ui.staging.american-science-cloud.org").rstrip("/")
GLOBUS_COLLECTIONS_PAGE = f"{RIG_UI_URL}/globus-collections"

# Literal Globus Transfer API version: part of the upstream Globus path, not an AmSC facility.
TRANSFER = f"{RIG_URL}/globus/external/transfer/v0.10"

HEADERS = {"Authorization": f"Bearer {PAT}", "Accept": "application/json"}
TIMEOUT = 60
POLL_INTERVAL = 5
POLL_TIMEOUT = 300

print("RIG                    :", RIG_URL)
print("Globus Collections page:", GLOBUS_COLLECTIONS_PAGE)

RIG                    : https://rig.staging.american-science-cloud.org
Globus Collections page: https://rig-ui.staging.american-science-cloud.org/globus-collections


### Helpers

In [3]:
def die(msg: str) -> None:
    print(f"\nERROR: {msg}")
    raise SystemExit(1)


def print_table(headers: List[str], rows: List[List[Any]]) -> None:
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    print(fmt.format(*headers))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()


def jwt_claims(token: str) -> dict:
    """Decode JWT claims for display only. Kong and RIG verify the signature."""
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return json.loads(base64.urlsafe_b64decode(payload))
    except Exception:
        return {}


def error_detail(r: requests.Response) -> str:
    """One-line summary of a RIG ({error}) or Globus ({code, message}) error body."""
    try:
        body = r.json()
    except ValueError:
        return r.text[:300]
    if isinstance(body, dict):
        parts = [str(body[k]) for k in ("error", "code", "message", "detail", "hint") if body.get(k)]
        if parts:
            return " | ".join(parts)[:400]
    return json.dumps(body)[:300]


def check(r: requests.Response, context: str) -> dict:
    """Return the JSON body, or stop with the failing status and error."""
    if not r.ok:
        die(f"{context} failed: HTTP {r.status_code} {error_detail(r)}")
    try:
        return r.json()
    except ValueError:
        return {}


def rig_get(url: str, context: str, **kwargs) -> dict:
    return check(requests.get(url, headers=HEADERS, timeout=TIMEOUT, **kwargs), context)


def rig_post(url: str, body: dict, context: str) -> dict:
    return check(requests.post(url, headers=HEADERS, json=body, timeout=TIMEOUT), context)


def join_remote(root: str, name: str) -> str:
    return root + name if root.endswith("/") else f"{root}/{name}"


def human_size(n: Any) -> str:
    try:
        v = float(n)
    except (TypeError, ValueError):
        return str(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if v < 1024 or unit == "TiB":
            return f"{int(v)} {unit}" if unit == "B" else f"{v:.1f} {unit}"
        v /= 1024

## 1) Load your AmSC PAT

This notebook does **not** log you in. Run **[`../login-amsc.ipynb`](../login-amsc.ipynb)** first: it opens MyAmSC, where you select a project
and create a PAT, and saves it to `~/.amsc_token.json`. The cell below loads it and shows which project RIG will use.

In [4]:
if not PAT:
    die(f"No PAT in AMSC_PAT or {home_path}. Run ../login-amsc.ipynb first.")

claims = jwt_claims(PAT)
MY_SUB = claims.get("sub")
PROJECT = claims.get("amsc_project_context")
exp = claims.get("exp")

print("sub                 :", MY_SUB)
print("amsc_project_context:", PROJECT)
print("expires             :", dt.datetime.fromtimestamp(exp, tz=dt.UTC).strftime("%Y-%m-%d %H:%M:%SZ") if exp else "unknown")

if exp and exp <= time.time():
    die("PAT expired. Run ../login-amsc.ipynb to get a new one.")
if not PROJECT or PROJECT == "undefined":
    die("PAT has no amsc_project_context. Select a project in MyAmSC and generate a new PAT.")

sub                 : b1a126d9-398c-40d9-97e8-4abb51a37ec5
amsc_project_context: amsc-rig-administration
expires             : 2026-10-28 21:50:36Z


## 2) Discover your project's Globus collections

`GET <RIG>/globus/internal/collections` returns the project's Globus Service Account identity and the collections
registered for the project. The Service Account can only reach these collections, and only under each registered path
(`normalized_path`) with the registered permissions (`r` or `rw`).

**No Service Account?** The VO / AmSC administrators provision one per project; ask them to set it up for your project.

**No collections?** Register one in RIG-UI → **Globus Collections** (link printed below), with the same project active
as in your PAT:
1. In the Globus web app, open the facility's mapped collection (or your Globus Connect Personal endpoint), choose
   **Add Guest Collection**, pick a directory, and create it. If your facility does not let users create guest
   collections, ask the facility administrators.
2. In the guest collection's permissions, add the project's **Service Account identity** (printed below, and shown on
   the Globus Collections page) with read or read/write access. Note the guest collection UUID.
3. In RIG-UI → Globus Collections → **Add my own Globus endpoint**, register that collection UUID and path.
   RIG checks that the Service Account can reach it before saving.
4. Keep the endpoint running (e.g. Globus Connect Personal) whenever transfers are expected.

Collections registered with visibility `self` by other project members are private to them and are not usable here.

> **[Screenshot placeholder] RIG-UI Globus Collections: Service Account identity and registered collections**  
> Save as `amsc/images/rig-ui/globus-collections-overview.png` and replace this block with  
> `![RIG-UI Globus Collections: Service Account identity and registered collections](../images/rig-ui/globus-collections-overview.png)`

> **[Screenshot placeholder] RIG-UI Globus Collections: Add my own Globus endpoint**  
> Save as `amsc/images/rig-ui/globus-collections-add.png` and replace this block with  
> `![RIG-UI Globus Collections: Add my own Globus endpoint](../images/rig-ui/globus-collections-add.png)`

In [5]:
body = rig_get(f"{RIG_URL}/globus/internal/collections", "list collections")
SA_IDENTITY = body.get("service_account_identity")
print("Project                  :", body.get("project"))
print("Service Account identity :", SA_IDENTITY or "(none)")

if not SA_IDENTITY:
    die(f"No Globus Service Account is provisioned for project {PROJECT!r}. Ask the VO / AmSC administrators to provision one.")

# Only collections this caller may use: shared with the project, or registered by this user.
collections = [
    c for c in body.get("collections", [])
    if not c.get("disabled_at") and (c.get("visibility") == "project" or c.get("owner_sub") == MY_SUB)
]
if not collections:
    die(f"No usable Globus collections for project {PROJECT!r}. Register one at {GLOBUS_COLLECTIONS_PAGE}")

print(f"\n=== GLOBUS COLLECTIONS (project {PROJECT}) ===")
print_table(
    ["#", "Label", "Collection ID", "Path", "Perms", "Direction", "Visibility", "Verified"],
    [
        [i, c.get("label"), c["collection_id"], c["normalized_path"], c["permissions"],
         c.get("direction"), c.get("visibility"), c.get("verification_status")]
        for i, c in enumerate(collections)
    ],
)


def pick_collection(title: str, candidates: List[dict], env_name: str, default: dict) -> dict:
    """Pick by env var or prompt: '#' from the table above, collection UUID, or label."""
    default_idx = collections.index(default)
    choice = os.environ.get(env_name) or input(f"{title} [# / collection id / label, default {default_idx}]: ").strip()
    if not choice:
        return default
    for i, c in enumerate(collections):
        if choice in (str(i), c["collection_id"], c.get("label")):
            if c not in candidates:
                die(f"Collection {c.get('label')!r} cannot be used as {title.lower()} (direction/permissions).")
            return c
    die(f"Unknown collection {choice!r}.")

Project                  : amsc-rig-administration
Service Account identity : 00e295ce-f6e5-452f-8400-98d5fbb9617a@clients.auth.globus.org

ERROR: No usable Globus collections for project 'amsc-rig-administration'. Register one at https://rig-ui.staging.american-science-cloud.org/globus-collections


SystemExit: 1

/Users/jbalcas/work/amsc/iri/iri-facility-api-examples/.venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 3) List files

A normal Globus Transfer `ls` (`GET /operation/endpoint/<collection>/ls?path=...`), sent through RIG with your PAT.
Pick the **source** collection; its registered path is listed.

In [ ]:
sources = [c for c in collections if c.get("direction") in ("source", "both", None)]
if not sources:
    die("No collection is registered with direction 'source' or 'both'.")
SOURCE = pick_collection("Source collection", sources, "GLOBUS_SOURCE", sources[0])


def globus_ls(collection: dict, path: str) -> List[dict]:
    body = rig_get(
        f"{TRANSFER}/operation/endpoint/{collection['collection_id']}/ls",
        f"ls {collection.get('label')!r}:{path}",
        params={"path": path},
    )
    return body.get("DATA", [])


items = globus_ls(SOURCE, SOURCE["normalized_path"])
print(f"\n{len(items)} item(s) under {SOURCE.get('label')!r}:{SOURCE['normalized_path']}")
print_table(
    ["Type", "Size", "Last modified", "Name"],
    [[it.get("type"), human_size(it.get("size", "")), it.get("last_modified", ""), it.get("name")] for it in items[:50]],
)
if len(items) > 50:
    print(f"... and {len(items) - 50} more")

## 4) Copy a file

Globus transfers are asynchronous: create the target directory (`mkdir`), fetch a one-time `submission_id`, submit a
`transfer` task, then poll it until it reaches `SUCCEEDED`.

The source file is `GLOBUS_FILENAME`, or the first non-empty file listed in step 3. The **destination** must be a
collection registered with `rw` permissions; by default it is another collection than the source when one exists,
otherwise the source collection itself. The copy goes into a new `amsc_rig_tutorial_<timestamp>/` directory.

In [ ]:
files = [it for it in items if it.get("type") == "file" and int(it.get("size") or 0) > 0]
FILENAME = os.environ.get("GLOBUS_FILENAME") or (files[0]["name"] if files else None)
if not FILENAME:
    die(f"No non-empty file under {SOURCE.get('label')!r}:{SOURCE['normalized_path']}. Stage a small file there, or set GLOBUS_FILENAME.")

destinations = [c for c in collections if "w" in c["permissions"] and c.get("direction") in ("destination", "both", None)]
if not destinations:
    die("No collection is registered with rw permissions and direction 'destination' or 'both'.")
default_dest = next((c for c in destinations if c["collection_id"] != SOURCE["collection_id"]), destinations[0])
DEST = pick_collection("Destination collection", destinations, "GLOBUS_DEST", default_dest)

run_ts = dt.datetime.now(dt.UTC).strftime("%Y%m%d%H%M%S")
src_path = join_remote(SOURCE["normalized_path"], FILENAME)
DEST_DIR = join_remote(DEST["normalized_path"], f"amsc_rig_tutorial_{run_ts}")
dst_path = join_remote(DEST_DIR, FILENAME)

print(f"\ncopy: {SOURCE.get('label')!r}:{src_path}")
print(f"  -> {DEST.get('label')!r}:{dst_path}\n")

rig_post(
    f"{TRANSFER}/operation/endpoint/{DEST['collection_id']}/mkdir",
    {"DATA_TYPE": "mkdir", "path": DEST_DIR},
    f"mkdir {DEST_DIR}",
)
print(f"created {DEST_DIR}")


def submission_id() -> str:
    return rig_get(f"{TRANSFER}/submission_id", "submission_id")["value"]


def poll(task_id: str) -> Dict[str, Any]:
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        task = rig_get(f"{TRANSFER}/task/{task_id}", f"task {task_id}")
        status = task.get("status")
        print(f"  task {task_id[:8]}...: status={status} bytes_transferred={task.get('bytes_transferred', 0)}")
        if status in ("SUCCEEDED", "FAILED", "CANCELED", "EXPIRED"):
            return task
        time.sleep(POLL_INTERVAL)
    die(f"Task {task_id} did not finish within {POLL_TIMEOUT}s.")


copy_task = rig_post(
    f"{TRANSFER}/transfer",
    {
        "DATA_TYPE": "transfer",
        "submission_id": submission_id(),
        "source_endpoint": SOURCE["collection_id"],
        "destination_endpoint": DEST["collection_id"],
        "DATA": [{"DATA_TYPE": "transfer_item", "source_path": src_path, "destination_path": dst_path, "recursive": False}],
        "label": f"AmSC RIG tutorial copy {run_ts}",
        "notify_on_succeeded": False,
        "notify_on_failed": False,
    },
    "submit transfer",
)["task_id"]
print(f"transfer task: {copy_task}")

task = poll(copy_task)
if task.get("status") != "SUCCEEDED":
    die(f"Transfer ended {task.get('status')}: {task.get('nice_status_details')}")

print()
for it in globus_ls(DEST, DEST_DIR):
    print(f"  {it.get('type'):9} {human_size(it.get('size', '')):>10}  {it.get('name')}")

## 5) Delete the copy

Globus delete is also an asynchronous task: fetch a `submission_id`, submit a `delete` task, and poll it. This removes
the `amsc_rig_tutorial_<timestamp>/` directory created in step 4 (recursively). Skip this cell to keep the copy.

In [ ]:
delete_task = rig_post(
    f"{TRANSFER}/delete",
    {
        "DATA_TYPE": "delete",
        "submission_id": submission_id(),
        "endpoint": DEST["collection_id"],
        "DATA": [{"DATA_TYPE": "delete_item", "path": DEST_DIR}],
        "label": f"AmSC RIG tutorial cleanup {run_ts}",
        "recursive": True,
    },
    f"delete {DEST_DIR}",
)["task_id"]
print(f"delete: {DEST.get('label')!r}:{DEST_DIR}")
print(f"delete task: {delete_task}")

task = poll(delete_task)
if task.get("status") != "SUCCEEDED":
    die(f"Delete ended {task.get('status')}: {task.get('nice_status_details')}")

## Recap & troubleshooting

You moved data **without ever holding a Globus token**: AmSC PAT → Kong → RIG (project from the PAT, swapped for the
project's Globus Service Account token) → Globus Transfer API.

| Symptom | Meaning | Fix |
|---|---|---|
| 401 from RIG | PAT expired, revoked, or from another deployment | Run [`../login-amsc.ipynb`](../login-amsc.ipynb) again |
| 400 `Cannot determine caller's active project` | PAT has no project | Select a project in MyAmSC and create a new PAT |
| 409 `No Globus Service Account provisioned for project ...` | Project has no Globus Service Account | Ask the VO / AmSC administrators to provision one |
| 500 `Service Account credential problem` | The project's Service Account secret is invalid or missing | Ask the VO / AmSC administrators |
| Globus `PermissionDenied` / `ClientError.Forbidden` | Path outside the registered collection path, or the Service Account lost access | Use a path under the collection's `normalized_path`; check the grant in Globus and in RIG-UI → Globus Collections |
| Transfer `FAILED` | Globus task-level error (e.g. source file missing) | Check `nice_status_details`; confirm `GLOBUS_FILENAME` exists on the source |

The same calls work outside this notebook: send your PAT as `Authorization: Bearer ...` to
`<RIG>/globus/external/transfer/v0.10/...` with any Globus Transfer API path. See [`README.md`](README.md) in this folder for `curl`
versions.